In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv("/content/pavement_transportation_master_dataset(4).csv")
target = "maintenance_cost_USD"
features = ["survey_year","lanes","length","speed_limit_kmh","section_age_years","AADT_veh_day","truck_percent","annual_rainfall_mm","mean_temp_C","freeze_thaw_cycles","subgrade_CBR_percent","pavement_thickness_mm","density_percent","binder_content_percent","IRI_m_per_km","rut_depth_mm","cracking_percent","FWD_deflection_mm","skid_resistance","avg_operating_speed_kmh","crash_count",target]

data = df[features].copy()
train, test = train_test_split(data, test_size=0.25, random_state=42)
train, test = train.copy(), test.copy()
test_true = test[target].copy()

rng = np.random.default_rng(42)
train_idx = rng.choice(train.index, round(0.05 * len(train)), replace=False)
test_idx = rng.choice(test.index, round(0.05 * len(test)), replace=False)
train.loc[train_idx, target] = np.nan
test.loc[test_idx, target] = np.nan

scalers = {"Min-Max": MinMaxScaler(), "Standard": StandardScaler(), "Robust": RobustScaler()}
rows = []

for scale_name, scaler in scalers.items():
    train_scaled = pd.DataFrame(scaler.fit_transform(train), index=train.index, columns=features)
    test_scaled = pd.DataFrame(scaler.transform(test), index=test.index, columns=features)
    j = features.index(target)
    if scale_name == "Min-Max":
        center, factor = scaler.data_min_[j], scaler.scale_[j]
        inverse = lambda x: np.asarray(x) / factor + center
    elif scale_name == "Standard":
        center, factor = scaler.mean_[j], scaler.scale_[j]
        inverse = lambda x: np.asarray(x) * factor + center
    else:
        center, factor = scaler.center_[j], scaler.scale_[j]
        inverse = lambda x: np.asarray(x) * factor + center

    for method in ["Mean", "Median", "KNN (k=5)"]:
        tr, te = train_scaled.copy(), test_scaled.copy()
        if method == "Mean":
            fill = tr[target].mean()
            tr[target] = tr[target].fillna(fill)
            te[target] = te[target].fillna(fill)
        elif method == "Median":
            fill = tr[target].median()
            tr[target] = tr[target].fillna(fill)
            te[target] = te[target].fillna(fill)
        else:
            imputer = KNNImputer(n_neighbors=5, weights="distance")
            tr = pd.DataFrame(imputer.fit_transform(tr), index=tr.index, columns=features)
            te = pd.DataFrame(imputer.transform(te), index=te.index, columns=features)
        y_pred = inverse(te.loc[test_idx, target])
        y_true = test_true.loc[test_idx].to_numpy()
        rows.append({"Scaling":scale_name,"Imputation":method,
                     "MAE_USD":mean_absolute_error(y_true,y_pred),
                     "RMSE_USD":np.sqrt(mean_squared_error(y_true,y_pred)),
                     "Bias_USD":np.mean(y_pred-y_true)})

results = pd.DataFrame(rows).sort_values(["MAE_USD","RMSE_USD"])
display(results.round(2))
plot_data = results.assign(Method=results["Scaling"]+" + "+results["Imputation"]).sort_values("MAE_USD")
plt.figure(figsize=(9,5))
plt.barh(plot_data["Method"],plot_data["MAE_USD"])
plt.xlabel("MAE (USD)")
plt.title("Imputation MAE Comparison")
plt.tight_layout()
plt.show()



In [1]:
# Results from the completed run


    Scaling Imputation  MAE_USD  RMSE_USD  Bias_USD
0   Min-Max     Median  3540.89   9569.78  -2688.47
1  Standard     Median  3540.89   9569.78  -2688.47
2    Robust     Median  3540.89   9569.78  -2688.47
3    Robust  KNN (k=5)  3850.37   8611.13   -249.21
4   Min-Max  KNN (k=5)  4071.76   9735.63   -940.76
5  Standard  KNN (k=5)  4089.71   8749.02   -532.79
6   Min-Max       Mean  4432.06   9185.26    126.93
7  Standard       Mean  4432.06   9185.26    126.93
8    Robust       Mean  4432.06   9185.26    126.93